# MLV Lineage and Refresh Monitoring Demo

**Author:** Meagan Longoria  
**Last Updated:** April 30, 2026  
**Blog Post:** [datasavvy.me](https://datasavvy.me/2026/04/30/programmatically-retrieving-mlv-lineage-and-refresh-times/)

---

## Overview

This notebook demonstrates how to programmatically retrieve lineage and refresh timestamps for materialized lake views (MLVs) in a Microsoft Fabric lakehouse. It builds a small demo environment from scratch, refreshes the MLVs, and visualizes the dependency graph with staleness highlighting using vis.js rendered via `displayHTML`.

The notebook covers:
- Creating base Delta tables with Change Data Feed enabled
- Creating MLVs with dependencies across multiple layers
- Parsing MLV lineage from the `fabric.source.entities` table property
- Retrieving last modified and last refresh timestamps from Delta transaction history
- Detecting stale MLVs and propagating staleness downstream through the dependency graph
- Rendering an interactive lineage diagram in the notebook

---

## Prerequisites

- A Microsoft Fabric workspace with a Fabric-enabled capacity
- A schema-enabled lakehouse attached as the default lakehouse for this notebook, with Contributor access or higher
- Fabric Runtime 1.3 or later (required for materialized lake views)
- Internet access from your Fabric capacity (required to load vis.js from CDN for diagram rendering)

---

## Configuration

Before running, set the following variables in **Cell 1**:

| Variable | Description | Default |
|---|---|---|
| `REFRESH_IN_NOTEBOOK` | If `True`, MLVs are refreshed via Spark SQL in this notebook. If `False`, refresh is skipped and should be managed from the Lakehouse MLV management area. | `True` |
| `SCHEMA_NAME` | The schema in the lakehouse where tables and MLVs will be created. | `dbo` |

---

## ⚠️ Warning

Running **Cell 4** (`setup_lakehouse()`) will drop and recreate all tables and MLVs listed in that function. Do not run this notebook against a lakehouse with existing objects you want to keep.

---

## Cell Guide

| Cell | Description |
|---|---|
| Cell 1 | Configuration and imports |
| Cell 2 | Function definitions — setup and refresh |
| Cell 3 | Function definitions — lineage, staleness, and diagram |
| Cell 4 | Step 1: Drop and recreate all tables and MLVs with sample data |
| Cell 5 | Step 2: Parse lineage and refresh MLVs |
| Cell 6 | Step 3: Render baseline lineage diagram |
| Cell 7 | Step 4: Insert new rows into table1 to simulate source data change |
| Cell 8 | Step 5: Re-fetch timestamps and render staleness diagram |


In [ ]:
# ============================================================
# Configuration
# ============================================================
# Set REFRESH_IN_NOTEBOOK = True to trigger MLV refresh via
# Spark SQL in this notebook. Set to False if you prefer to
# manage refresh from the Lakehouse MLV management area.

REFRESH_IN_NOTEBOOK = True
SCHEMA_NAME = "dbo"

# ============================================================
# Imports
# ============================================================

import json
from datetime import datetime
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

spark = SparkSession.builder.getOrCreate()

print("Config and imports ready.")

In [ ]:
# ============================================================
# Function definitions — Part 1
# setup_lakehouse, refresh_mlvs
# ============================================================

# ------------------------------------------------------------
# Lakehouse setup: drop and recreate all tables and MLVs
# ------------------------------------------------------------

def setup_lakehouse():
    # Drop MLVs in reverse dependency order before dropping base tables
    for mlv in ["mlvc", "mlvb", "mlva"]:
        spark.sql(f"DROP MATERIALIZED LAKE VIEW IF EXISTS {mlv}")
        print(f"Dropped (if existed): {mlv}")

    for tbl in ["table1", "table2", "table3"]:
        spark.sql(f"DROP TABLE IF EXISTS {tbl}")
        print(f"Dropped (if existed): {tbl}")

    # Create base tables as empty Delta tables
    spark.createDataFrame([], schema=StructType([
        StructField("id", IntegerType(), False),
        StructField("name", StringType(), True),
        StructField("value", IntegerType(), True),
    ])).write.format("delta").mode("overwrite").saveAsTable("table1")
    print("Created: table1")

    spark.createDataFrame([], schema=StructType([
        StructField("id", IntegerType(), False),
        StructField("category", StringType(), True),
        StructField("amount", IntegerType(), True),
    ])).write.format("delta").mode("overwrite").saveAsTable("table2")
    print("Created: table2")

    spark.createDataFrame([], schema=StructType([
        StructField("id", IntegerType(), False),
        StructField("description", StringType(), True),
        StructField("status", StringType(), True),
    ])).write.format("delta").mode("overwrite").saveAsTable("table3")
    print("Created: table3")

    # Enable CDF on base tables to support optimal incremental refresh
    for tbl in ["table1", "table2", "table3"]:
        spark.sql(f"ALTER TABLE {tbl} SET TBLPROPERTIES ('delta.enableChangeDataFeed' = 'true')")
        print(f"CDF enabled: {tbl}")

    # Insert initial sample data
    spark.sql("""
        INSERT INTO table1 (id, name, value) VALUES
        (1, 'Alpha',   100), (2, 'Beta',    200), (3, 'Gamma',   300),
        (4, 'Delta',   400), (5, 'Epsilon', 500), (6, 'Zeta',    600),
        (7, 'Eta',     700)
    """)
    print("Inserted sample data: table1")

    spark.sql("""
        INSERT INTO table2 (id, category, amount) VALUES
        (1, 'CategoryA', 1000), (2, 'CategoryB', 2000), (3, 'CategoryC', 3000),
        (4, 'CategoryA', 4000), (5, 'CategoryB', 5000), (6, 'CategoryC', 6000),
        (7, 'CategoryA', 7000)
    """)
    print("Inserted sample data: table2")

    spark.sql("""
        INSERT INTO table3 (id, description, status) VALUES
        (1, 'First record',   'Active'),   (2, 'Second record',  'Inactive'),
        (3, 'Third record',   'Active'),   (4, 'Fourth record',  'Pending'),
        (5, 'Fifth record',   'Active'),   (6, 'Sixth record',   'Inactive'),
        (7, 'Seventh record', 'Active')
    """)
    print("Inserted sample data: table3")

    # Create MLVs with CDF enabled
    # MLVA: sourced from table1
    spark.sql("""
        CREATE OR REPLACE MATERIALIZED LAKE VIEW mlva
        TBLPROPERTIES ('delta.enableChangeDataFeed' = 'true')
        AS SELECT id, name, value FROM table1
    """)
    print("Created: mlva")

    # MLVB: sourced from table1 and table2
    spark.sql("""
        CREATE OR REPLACE MATERIALIZED LAKE VIEW mlvb
        TBLPROPERTIES ('delta.enableChangeDataFeed' = 'true')
        AS
        SELECT t1.id, t1.name, t2.category, t2.amount
        FROM table1 t1 INNER JOIN table2 t2 ON t1.id = t2.id
    """)
    print("Created: mlvb")

    # MLVC: sourced from mlva, mlvb, and table3
    spark.sql("""
        CREATE OR REPLACE MATERIALIZED LAKE VIEW mlvc
        TBLPROPERTIES ('delta.enableChangeDataFeed' = 'true')
        AS
        SELECT a.id, a.name, a.value, b.category, b.amount, t3.description, t3.status
        FROM mlva a
        INNER JOIN mlvb b ON a.id = b.id
        INNER JOIN table3 t3 ON a.id = t3.id
    """)
    print("Created: mlvc")

    # Verify CDF and row counts across all objects
    print("\n--- Verification ---")
    for obj in ["table1", "table2", "table3", "mlva", "mlvb", "mlvc"]:
        props = spark.sql(f"SHOW TBLPROPERTIES {obj}").collect()
        cdf_enabled = any(
            row["key"] == "delta.enableChangeDataFeed" and row["value"] == "true"
            for row in props
        )
        print(f"{obj} — CDF enabled: {cdf_enabled}, Row count: {spark.table(obj).count()}")

# ------------------------------------------------------------
# MLV refresh: topological sort ensures upstream MLVs refresh
# before downstream dependents
# ------------------------------------------------------------

def refresh_mlvs(schema_name, lineage, refresh_in_notebook):
    if not refresh_in_notebook:
        print("REFRESH_IN_NOTEBOOK is False — skipping notebook refresh.")
        print("Manage MLV refresh from the Lakehouse MLV management area.")
        return

    def get_refresh_order(lineage):
        """Topological sort of MLVs by dependency."""
        visited = []
        def visit(node):
            if node in lineage and node not in visited:
                for src in lineage[node]:
                    visit(src)
                visited.append(node)
        for mlv in lineage:
            visit(mlv)
        return visited

    refresh_order = get_refresh_order(lineage)
    print(f"Refresh order: {refresh_order}")
    for mlv in refresh_order:
        print(f"Refreshing {schema_name}.{mlv}...")
        spark.sql(f"REFRESH MATERIALIZED LAKE VIEW {schema_name}.{mlv} FULL")
        print(f"Refreshed: {mlv}")

print("Functions defined: setup_lakehouse, refresh_mlvs")

In [ ]:
# ============================================================
# Function definitions — Part 2
# get_lineage_and_timestamps, get_stale_nodes,
# render_diagram, insert_new_data
# ============================================================

# ------------------------------------------------------------
# Lineage parsing: get MLV sources and last modified timestamps
# for all tables and MLVs from Delta history
# ------------------------------------------------------------

def get_lineage_and_timestamps(schema_name):
    mlvs = [
        row["name"]
        for row in spark.sql(f"SHOW MATERIALIZED LAKE VIEWS IN {schema_name}").collect()
    ]
    print(f"Found MLVs: {mlvs}")

    lineage = {}
    last_modified = {}

    all_objects = [row["tableName"] for row in spark.sql("SHOW TABLES").collect()]
    for obj in all_objects:
        try:
            # LIMIT 1 returns the most recent Delta transaction log entry
            history = spark.sql(f"DESCRIBE HISTORY {schema_name}.{obj} LIMIT 1").collect()
            if history:
                ts = history[0]["timestamp"]
                last_modified[obj] = (
                    ts.strftime("%Y-%m-%d %H:%M") if isinstance(ts, datetime) else str(ts)[:16]
                )
            else:
                last_modified[obj] = "unknown"
        except Exception:
            last_modified[obj] = "unknown"
        print(f"{obj} last modified: {last_modified[obj]}")

    # Parse lineage from fabric.source.entities tblproperty
    # which Fabric populates automatically on MLV creation/refresh
    for mlv in mlvs:
        props = spark.sql(f"SHOW TBLPROPERTIES {schema_name}.{mlv}").collect()
        source_entities_raw = next(
            (row["value"] for row in props if row["key"] == "fabric.source.entities"),
            None
        )
        sources = (
            [e["tableName"] for e in json.loads(source_entities_raw)]
            if source_entities_raw else []
        )
        lineage[mlv] = sources
        print(f"{mlv} <- {sources}")

    return lineage, last_modified

# ------------------------------------------------------------
# Staleness detection: an MLV is stale if any upstream source
# has a newer timestamp, and staleness propagates downstream
# ------------------------------------------------------------

def get_stale_nodes(lineage, last_modified):
    def parse_ts(ts_str):
        """Parse timestamp string to datetime for comparison."""
        if not ts_str or ts_str == "unknown":
            return None
        try:
            return datetime.strptime(ts_str[:16], "%Y-%m-%d %H:%M")
        except ValueError:
            return None

    def get_all_downstream(node, visited=None):
        """Recursively find all nodes that depend on the given node."""
        if visited is None:
            visited = set()
        for mlv, srcs in lineage.items():
            if node in srcs and mlv not in visited:
                visited.add(mlv)
                get_all_downstream(mlv, visited)
        return visited

    stale_nodes = set()
    for mlv, srcs in lineage.items():
        mlv_ts = parse_ts(last_modified.get(mlv))
        if mlv_ts is None:
            continue
        for src in srcs:
            src_ts = parse_ts(last_modified.get(src))
            if src_ts and src_ts > mlv_ts:
                # Mark this MLV and everything downstream as stale
                stale_nodes.add(mlv)
                stale_nodes.update(get_all_downstream(mlv))

    print(f"Stale nodes: {stale_nodes if stale_nodes else 'none'}")
    return stale_nodes

# ------------------------------------------------------------
# Diagram renderer: builds and displays a vis.js lineage diagram
# via displayHTML using vis.js loaded from CDN.
#
# Node colors:
#   Green  = base Delta table (up to date)
#   Blue   = materialized lake view (up to date)
#   Orange = object that needs a refresh
#
# Edge colors:
#   Gray   = no staleness on this path
#   Orange = source is newer than the target MLV
#
# Edges are only colored orange if the source itself is the
# cause of staleness — edges from up-to-date sources into a
# stale MLV remain gray.
#
# Selection behavior is overridden via the `chosen` property
# to prevent vis.js from applying its default blue outline on
# node selection, and edge highlight colors are explicitly set
# to match the edge color so orange edges stay orange when a
# connected node is selected.
#
# Pass stale_nodes to enable staleness highlighting; omit or
# pass None for the plain lineage view.
# ------------------------------------------------------------

def render_diagram(lineage, last_modified, stale_nodes=None, title="MLV Lineage Diagram"):
    if stale_nodes is None:
        stale_nodes = set()

    last_modified_json = json.dumps(last_modified)
    lineage_json = json.dumps(lineage)
    stale_nodes_json = json.dumps(list(stale_nodes))

    html = f"""
<!DOCTYPE html>
<html>
<head>
    <script src="https://cdn.jsdelivr.net/npm/vis-network@9.1.9/dist/vis-network.min.js"></script>
    <link href="https://cdn.jsdelivr.net/npm/vis-network@9.1.9/dist/vis-network.min.css" rel="stylesheet"/>
    <style>
        #diagram {{ width: 100%; height: 500px; background: #f9f9f9; border-radius: 8px; }}
        body {{ font-family: Segoe UI, sans-serif; margin: 0; padding: 10px; }}
        h3 {{ margin-bottom: 6px; }}
        .legend {{ display: flex; gap: 20px; margin-bottom: 10px; font-size: 13px; align-items: center; }}
        .legend-dot {{ width: 14px; height: 14px; border-radius: 3px; display: inline-block; margin-right: 5px; vertical-align: middle; }}
    </style>
</head>
<body>
<h3>{title}</h3>
<div class="legend">
    <span><span class="legend-dot" style="background:#2d8a53"></span>Base Table</span>
    <span><span class="legend-dot" style="background:#2d6bba"></span>Materialized Lake View</span>
    <span><span class="legend-dot" style="background:#c45200"></span>Needs Refresh</span>
</div>
<div id="diagram"></div>
<script>
    const lastModified = {last_modified_json};
    const lineageData = {lineage_json};
    const staleNodes  = new Set({stale_nodes_json});

    const mlvSet = new Set(Object.keys(lineageData));
    const allSources = new Set();
    Object.values(lineageData).forEach(srcs => srcs.forEach(s => allSources.add(s)));
    const allNodes = [...new Set([...mlvSet, ...allSources])];

    function nodeColors(name) {{
        if (staleNodes.has(name)) {{
            return {{ background: "#c45200", border: "#9e4200", highlight: {{ background: "#9e4200" }} }};
        }}
        if (mlvSet.has(name)) {{
            return {{ background: "#2d6bba", border: "#245496", highlight: {{ background: "#245496" }} }};
        }}
        return {{ background: "#2d8a53", border: "#246e42", highlight: {{ background: "#246e42" }} }};
    }}

    const nodes = allNodes.map(name => ({{
        id: name,
        label: name + "\\n" + (lastModified[name] || ""),
        color: nodeColors(name),
        font: {{ color: "white", size: 13 }},
        shape: "box",
        margin: 10,
        shadow: true,
        chosen: {{
            node: function(values, id, selected, hovering) {{
                values.borderColor = nodeColors(id).border;
                values.borderWidth = 2;
            }}
        }}
    }}));

    // Color an edge orange only if the source is stale or newer than
    // the target — avoids falsely highlighting edges from up-to-date sources
    const edges = [];
    Object.entries(lineageData).forEach(([mlv, srcs]) => {{
        srcs.forEach(src => {{
            const isStaleEdge = staleNodes.has(src) ||
                (staleNodes.has(mlv) && !mlvSet.has(src) &&
                 lastModified[src] > lastModified[mlv]);
            edges.push({{
                from: src,
                to: mlv,
                arrows: "to",
                color: {{
                    color: isStaleEdge ? "#c45200" : "#555",
                    highlight: isStaleEdge ? "#c45200" : "#555"
                }},
                width: isStaleEdge ? 3 : 2,
                smooth: {{ type: "cubicBezier", forceDirection: "horizontal" }}
            }});
        }});
    }});

    const container = document.getElementById("diagram");
    const data = {{ nodes: new vis.DataSet(nodes), edges: new vis.DataSet(edges) }};
    const options = {{
        layout: {{ hierarchical: {{ direction: "LR", sortMethod: "directed", levelSeparation: 180, nodeSpacing: 100 }} }},
        physics: {{ enabled: false }},
        edges: {{ width: 2 }},
        interaction: {{ dragNodes: true, zoomView: true }}
    }};
    new vis.Network(container, data, options);
</script>
</body>
</html>
"""
    displayHTML(html)

print("Functions defined: get_lineage_and_timestamps, get_stale_nodes, render_diagram, insert_new_data")

# ------------------------------------------------------------
# New data insertion: adds rows to table1 to simulate
# source data changes for staleness demonstration
# ------------------------------------------------------------

def insert_new_data():
    spark.sql("""
        INSERT INTO table1 (id, name, value) VALUES
        (8,  'Theta', 800),
        (9,  'Iota',  900),
        (10, 'Kappa', 1000)
    """)
    print("Inserted 3 new rows into table1 (ids 8, 9, 10)")

print("Functions defined: get_lineage_and_timestamps, get_stale_nodes, render_diagram, insert_new_data")

In [ ]:
# ============================================================
# Step 1: Drop and recreate all tables and MLVs with sample data
# ============================================================

setup_lakehouse()

In [ ]:
# ============================================================
# Step 2: Parse lineage then refresh MLVs.
# Controlled by REFRESH_IN_NOTEBOOK in Cell 1.
# If False, go to the Lakehouse MLV management area to refresh.
# ============================================================

lineage, last_modified = get_lineage_and_timestamps(SCHEMA_NAME)
refresh_mlvs(SCHEMA_NAME, lineage, REFRESH_IN_NOTEBOOK)

In [ ]:
# ============================================================
# Step 3: Render baseline lineage diagram — no staleness
# highlighting, just the current pipeline structure and
# last modified timestamps for all objects
# ============================================================

render_diagram(lineage, last_modified, title="MLV Lineage — Current State")

In [ ]:
# ============================================================
# Step 4: Insert new rows into table1 to simulate source
# data arriving after the last MLV refresh
# ============================================================

insert_new_data()

In [ ]:
# ============================================================
# Step 5: Re-fetch timestamps to pick up the new table1 write,
# then render the lineage diagram with staleness highlighting
# to show which MLVs are now out of date
# ============================================================

lineage, last_modified = get_lineage_and_timestamps(SCHEMA_NAME)
stale_nodes = get_stale_nodes(lineage, last_modified)
render_diagram(lineage, last_modified, stale_nodes=stale_nodes, title="MLV Lineage — Staleness Check")